# Human vs Machine Translation Detection Codebase

This Jupyter Notebook is designed to prepare a dataset for detecting human versus machine-generated translations, focusing on English to Afrikaans translations. It leverages the SMOL dataset from Hugging Face to extract human translations and integrates machine translation outputs from text files.

## Key Steps:
1. **Setup and Configuration**: Import necessary libraries and define parameters such as the dataset configuration, output directory, and number of rows to process.
2. **Data Loading and Cleaning**: Define functions to load and clean the dataset, ensuring valid human translations and source texts are extracted.
3. **CSV Generation**: Load the specified number of rows from the dataset and save them to a CSV file, including columns for human translations, source texts, and source language.
4. **Integration of Machine Translations**: Provide utilities to read machine translation outputs from text files and append them as new columns to the existing CSV.
5. **Execution**: Run the processes to generate the final CSV with both human and machine translations for analysis.

This notebook facilitates the creation of a balanced dataset for training and evaluating models to distinguish between human and machine translations.

In [ ]:
import os
import pandas as pd
from datasets import load_dataset

SMOL_DATASET = "google/smol"
OUTPUT_DIR = "smol_csvs/af"
ROWS = 500

os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG = "smolsent__en_af" # Change this to the desired language pair, e.g., "smolsent__en_sw" for English to Swahili
SOURCE_LANGUAGE_NAME = "English"

In [ ]:
def clean_df(df: pd.DataFrame, n: int, config_name: str) -> pd.DataFrame:
    df = df.dropna(subset=["human translation", "source text"]).copy()

    df["human translation"] = df["human translation"].astype(str).str.strip()
    df["source text"] = df["source text"].astype(str).str.strip()

    df = df[
        (df["human translation"] != "") &
        (df["source text"] != "")
    ].reset_index(drop=True)

    if len(df) < n:
        raise ValueError(
            f"After cleaning, {config_name} has only {len(df)} valid rows; expected {n}."
        )

    return df.iloc[:n].copy()
def load_english_to_swahili_rows(config_name: str, n: int = 500) -> pd.DataFrame:
    try:
        ds = load_dataset(SMOL_DATASET, config_name, split="train")
    except Exception as e:
        raise RuntimeError(f"Failed to load {config_name}: {e}")

    if len(ds) < n:
        raise ValueError(f"{config_name} only has {len(ds)} rows, but {n} were requested.")

    required_cols = {"src", "trg"}
    missing = required_cols - set(ds.column_names)
    if missing:
        raise ValueError(
            f"{config_name} is missing expected columns: {missing}. "
            f"Available columns: {ds.column_names}"
        )

    ds_small = ds.select(range(n))

    df = pd.DataFrame({
        "human translation": ds_small["trg"],              # Swahili/ Afrikaans
        "source text": ds_small["src"],                    # English
        "source language": [SOURCE_LANGUAGE_NAME] * n,     # English
    })

    return clean_df(df, n, config_name)

In [ ]:
output_file = os.path.join(OUTPUT_DIR, "") # Set the desired output file name, e.g., "english_to_afrikaans.csv"
df = load_english_to_swahili_rows(CONFIG, ROWS)
df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved {len(df)} rows to: {output_file}")
print("\nCounts by source language:")
print(df["source language"].value_counts())
print("\nDone.")

In [ ]:
import os
import pandas as pd


def extract_column_name(filename):
    """
    Extract column name from filename.
    Example: openwho_test_gemini.txt -> gemini
    """
    base = os.path.basename(filename)
    name = os.path.splitext(base)[0] 
    return name.split("_")[-1]       


def read_txt_file(filepath):
    """
    Read txt file and return cleaned lines.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]
    
    # Remove empty lines if any
    lines = [line for line in lines if line]
    return lines


def add_txt_columns_to_csv(csv_path, txt_folder, output_path):
    # Load CSV
    df = pd.read_csv(csv_path)
    num_rows = len(df)

    print(f"CSV loaded with {num_rows} rows")

    # Process each txt file
    for file in os.listdir(txt_folder):
        if file.endswith(".txt"):
            file_path = os.path.join(txt_folder, file)

            column_name = extract_column_name(file)
            lines = read_txt_file(file_path)

            print(f"Processing {file} → column '{column_name}'")

            # Validate length
            if len(lines) != num_rows:
                raise ValueError(
                    f"Row mismatch in {file}: "
                    f"{len(lines)} lines vs {num_rows} CSV rows"
                )

            # Add column
            df[column_name] = lines

    # Save new CSV
    df.to_csv(output_path, index=False)
    print(f"\nSaved updated CSV to: {output_path}")

In [ ]:
csv_file = "" # Set the path to generated CSV file above
txt_folder = " " # Set the path to the folder containing the txt files with machine translation outputs
output_file = " output.csv" # Set the path for the output CSV file with all translations

add_txt_columns_to_csv(csv_file, txt_folder, output_file)